##**[Wine pytorch]**

전복의 성별('M', 'F', 'I')을 예측

> 성별 분류는 특히 '유체(Infant)'와 '성체(Male/Female)' 간의 물리적 차이를 잘 구분하느냐가 핵심


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [13]:
# Wine 데이터셋 로드 (13개 특성, 3개 클래스)
wine = load_wine()
X, y = wine.data, wine.target

# 학습용/테스트용 데이터 분할 (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
# Z-Score 표준화 적용해보기 ( 데이터 정규화 )
# 특성들의 수치 범위를 평균 0, 표준편차 1로 맞추어 경사하강법의 학습 속도와 안정성 최적화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [15]:

# PyTorch 연산을 위해 Tensor 변환 (기본 자료형: float32 / 정답 라벨은 정수형: long)
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

In [14]:
# 2. 모델 아키텍처 설계 (Model Definition)

class WineClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(WineClassifier, self).__init__()
        # 입력층 -> 은닉층 1
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU() # 기울기 소실 문제를 방지하는 ReLU 활성화 함수

        # [과잉적합 방지] 드롭아웃 레이어 추가 (학습 시 20% 뉴런을 무작위로 비활성화)
        self.dropout = nn.Dropout(0.2)

        # 은닉층 1 -> 은닉층 2
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)

        # 은닉층 2 -> 출력층
        self.fc3 = nn.Linear(hidden_dim // 2, output_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        # [시험 꿀팁] PyTorch의 CrossEntropyLoss는 내부적으로 Softmax 연산을 포함하므로
        # 모델의 맨 마지막 출력에는 활성화 함수(Softmax)를 명시적으로 걸어주지 않습니다.
        return out

# 모델 인스턴스 생성 (입력 특성 13개, 은닉 노드 64개, 출력 클래스 3개)
model = WineClassifier(input_dim=13, hidden_dim=64, output_dim=3)

In [20]:
# 3. 오차 분석 도구 및 나침반 설정 (Loss & Optimizer)

# 다중 분류를 위한 교차 엔트로피 손실 함수 (내부적으로 소프트맥스 연산 자동 수행)
criterion = nn.CrossEntropyLoss()

# 경사하강법을 신경망에 적용한 Adam 옵티마이저 (학습률 0.01)
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [22]:
# 4. 모델 학습 진행 (Training Loop)
epochs = 100


for epoch in range(1, epochs + 1):
    model.train() # 모델을 학습 모드로 설정 (드롭아웃 활성화)

    # 순전파 (Forward Propagation): 예측값 계산
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # 역전파 (Backpropagation) 및 가중치 업데이트
    optimizer.zero_grad() # 기울기 누적 방지를 위해 이전 기울기 초기화
    loss.backward()       # 체인 룰을 이용해 각 가중치들의 기울기(오차) 역전파 계산
    optimizer.step()      # 가중치 최적화 업데이트

    # 10 에포크마다 중간 결과 출력
    if epoch % 10 == 0:
        print(f"Epoch [{epoch}/{epochs}] - Train Loss: {loss.item():.4f}")



Epoch [10/100] - Train Loss: 3.5411
Epoch [20/100] - Train Loss: 1.5338
Epoch [30/100] - Train Loss: 1.0472
Epoch [40/100] - Train Loss: 1.0634
Epoch [50/100] - Train Loss: 1.0091
Epoch [60/100] - Train Loss: 1.0022
Epoch [70/100] - Train Loss: 0.8923
Epoch [80/100] - Train Loss: 0.8589
Epoch [90/100] - Train Loss: 0.8567
Epoch [100/100] - Train Loss: 0.8256


In [23]:
# 5. 모델 평가 (Evaluation)

model.eval() # 모델을 평가 모드로 설정 (드롭아웃 비활성화)

with torch.no_grad(): # 테스트 단계에서는 역전파 기울기 계산을 생략하여 메모리 절약
    test_outputs = model(X_test_tensor)

    # [소프트맥스 확률 해석] 가장 큰 점수(로짓)를 가진 인덱스(0, 1, 2)를 예측 클래스로 결정
    _, predicted = torch.max(test_outputs, 1)

    # 정확도 계산
    correct = (predicted == y_test_tensor).sum().item()
    total = y_test_tensor.size(0)
    accuracy = (correct / total) * 100

    print("\n" + "="*30)
    print(f"Test Accuracy: {accuracy:.2f}%")
    print("="*30)

    # ===========================


Test Accuracy: 63.89%
